# YOLO11s + 공통 ResNet50 · v2 val e2e 평가

이번에는 모델을 새로 학습하지 않습니다. 학습한 YOLO가 찾은 박스를 자르고, 팀 공통 분류기로 어떤 약품인지 예측한 뒤 정답과 비교합니다.

**흐름:** 원본 이미지 → YOLO 예측 박스 → 기존 전처리로 crop → 공통 ResNet50 → 약품별 bbox·점수 → 클래스별 AP 평가.

Colab GPU에서 순서대로 실행하세요. 모델·데이터 파일은 Drive에 있는 경로를 사용합니다. 코드는 아래 셀에서 저장소를 가져옵니다.

**팀과 아직 확인할 부분:** 제공된 기존 노트북은 추론용이며 팀 공통 채점 코드는 포함하지 않습니다. 여기의 점수는 reference COCO 평가입니다. score 0.25를 탐지 점수에 적용한 결과와 최종 점수에 적용한 결과를 모두 기록합니다. 팀 채점 코드와 threshold 위치를 확인한 후 같은 조건의 값을 비교하세요.


In [ ]:
import os, sys, subprocess
os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_CONFIG_DIR"] = "/content/yolo-e2e-settings"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.183", "torchmetrics[detection]==1.9.0", "pycocotools",
    "pandas", "matplotlib", "pyyaml", "pillow"])
import torch
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0), torch.__version__)


## 저장소 코드 준비·Drive 연결

저장소에서 직접 실행합니다. Colab에 체크아웃이 없으면 GitHub에서 가져옵니다. PR 병합 전 검토 시 SOURCE_REF를 작업 브랜치로 지정하세요.


In [ ]:
import sys, subprocess
from pathlib import Path
SOURCE_REF = "main"  # PR 검토 중에는 "37-yolo11s-training-e2e"로 지정
REPOSITORY_URL = "https://github.com/auranova1125/gitkk-pill-detection.git"
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file()), None)
if project_root is None:
    project_root = Path("/content/gitkk-pill-detection")
    if not (project_root / ".git").is_dir():
        subprocess.check_call(["git", "clone", "--branch", SOURCE_REF, REPOSITORY_URL, str(project_root)])
PROJECT_ROOT = project_root.resolve()
assert (PROJECT_ROOT / "pipeline/yolo_e2e.py").is_file(), "YOLO 코드가 있는 브랜치를 열어주세요. PR 검토 중에는 SOURCE_REF를 변경하세요."
import os
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
# 이전 프로젝트 복사본이 같은 Colab 세션에서 import된 경우 새 체크아웃 기준으로 읽습니다.
for name, module in list(sys.modules.items()):
    filename = str(getattr(module, "__file__", ""))
    if name.split(".")[0] in {"pipeline", "classifier", "detector", "data_loading", "configs", "utils"}:
        if "/content/" in filename and not filename.startswith(str(PROJECT_ROOT)):
            del sys.modules[name]
from google.colab import drive
drive.mount("/content/drive")


## 2. 파일 경로와 실행 설정
경로는 이전 YOLO 학습 결과와 팀 공통 분류기를 가리킵니다. 가중치와 class map의 SHA-256을 대조하므로 같은 이름의 다른 파일을 쓰면 중단합니다.

`threshold_on="detector"`는 기존 통합 추론 코드의 동작을 따른 주 표기값입니다. 팀 문서의 의미는 아직 확인 중이므로 최종 점수 기준 결과도 함께 만듭니다. 기존 추론 예제에 남아 있는 0.05는 사용하지 않고, 이번 팀 문서의 **0.25**를 따릅니다.

원본 이미지 전체 17GB를 다시 복사하지 않습니다. 데이터 패키지는 Drive에서 검증하고, **평가에 필요한 val 이미지만 로컬에 복사**합니다. 이미지별 예측은 Drive에 저장돼 같은 설정으로 다시 실행하면 미완료 이미지부터 이어갑니다. 재연결할 때 같은 `E2E_RUN_NAME`을 사용하세요.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
ARCHIVE = Path("/content/drive/MyDrive/gitkk_archive")
DETECTOR_RUN = "20261103-yolo11s-det-v2-baseline-01"
E2E_RUN_NAME = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d") + "-yolo11s-resnet50-det-v2-evaluation-01"
# 재개할 때 위 줄을 기존 실행명 문자열로 바꿉니다. 예: E2E_RUN_NAME = "20261104-yolo11s-resnet50-det-v2-evaluation-01"
config = {
    "run_name": E2E_RUN_NAME,
    "detector_run": DETECTOR_RUN,
    "detector_checkpoint": str(ARCHIVE / "outputs/detector" / DETECTOR_RUN / "weights/best.pt"),
    "classifier_checkpoint": str(ARCHIVE / "models/classifier/resnet50_baseline_v2/best_v2_slim.pt"),
    "class_map": str(ARCHIVE / "models/classifier/resnet50_baseline_v2/class_map.json"),
    "package_dir": str(ARCHIVE / "detector_data/1. selected_data/v2"),
    "output_dir": str(ARCHIVE / "outputs/inference"),
    "image_cache_dir": "/content/e2e-v2-val-images",
    "device": "cuda:0", "detector_batch": 8, "classifier_batch": 64,
    "threshold_on": "detector", "limit": None,
}
config.update({
    "expected_classifier_sha256": "d32d1f3030fe14ba5edf6b18e048f81cc2377e735ba61d22d3c87a7e97d90d69",
    "expected_class_map_sha256": "166ed118a34efce825a7999f4e85f2aa8746be6ebfeedd7afca08651f7fb2407",
    "expected_detector_sha256": "754e8e46b61a3aa372cc78e7cca2627fafbc44c6b377a74f08f73e681623ce92",
})
for key in ("detector_checkpoint", "classifier_checkpoint", "class_map"):
    path = Path(config[key])
    assert path.is_file(), f"파일을 찾을 수 없습니다: {path}"
    print(key, path.name, f"{path.stat().st_size / 1024**2:.2f} MiB")
assert Path(config["package_dir"]).is_dir()
Path("outputs/runtime").mkdir(parents=True, exist_ok=True)
Path("outputs/runtime/e2e-config.json").write_text(json.dumps(config, indent=2))
print("실행명:", E2E_RUN_NAME)
print("최종 결과 폴더:", Path(config["output_dir"]) / E2E_RUN_NAME)


## 3. 먼저 이미지 3장으로 연결 확인
실제 YOLO와 공통 분류기를 불러옵니다. 정답 박스가 아닌 **YOLO 예측 박스**를 분류기에 넣습니다. 작은 표본의 점수는 작동 확인용이며 전체 성능으로 보고하지 않습니다.

처음에는 Drive 패키지의 train/val 무결성을 함께 확인하므로 시간이 걸릴 수 있습니다. 분류기 파일을 처음 읽는 시간도 포함됩니다.


In [ ]:
from pipeline.yolo_e2e import run_e2e
smoke_config = dict(config, run_name=config["run_name"].replace("-evaluation-", "-smoke-"), limit=3)
smoke_dir = run_e2e(smoke_config)
import pandas as pd
from IPython.display import display
preview = pd.read_csv(smoke_dir / "val_predictions.csv")
class_map = json.loads(Path(config["class_map"]).read_text())
names = {entry["category_id"]: entry["name"] for entry in class_map["classes"]}
preview["drug_name"] = preview["category_id"].map(names)
display(preview[["image_id", "category_id", "drug_name", "detector_score", "classifier_score", "score"]])
print("3장 연결 확인 완료. 이 표본 점수는 전체 val 성능이 아닙니다.")


## 4. 전체 2,026장 추론·채점
분류기를 학습하지 않으며 정답 품목은 채점에만 사용합니다. Crop은 기존 저장소의 여백 5% → 정사각형 padding → 224 resize → ImageNet normalization을 그대로 사용합니다. 최종 점수는 탐지 score와 분류 확률의 곱입니다.


메모리 부족으로 중단되면 detector_batch나 classifier_batch를 줄여 **새 실행 이름**으로 시작하세요. 같은 실행의 설정이나 파일이 바뀌면 기존 cache와 섞지 않도록 중단합니다.


In [ ]:
run_dir = run_e2e(config)
print("e2e 평가 완료:", run_dir)


## 5. 결과 확인·그래프·노션 문서 (런타임 초기화 후에도 독립 실행 가능)
이 셀은 Drive만 연결하면 사용할 수 있습니다. `RUN_NAME_TO_VIEW`를 비우면 해당 이름 패턴의 완료된 전체 실행 중 이름이 가장 뒤인 폴더를 고릅니다. 특정 결과를 보려면 정확한 실행 이름을 넣으세요. 데이터 복사나 모델 로딩은 필요 없습니다.


In [ ]:
from pathlib import Path
from google.colab import drive, files
from IPython.display import display, Markdown
import json, pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gitkk_archive/outputs/inference")
RUN_NAME_TO_VIEW = ""  # 예: "20261104-yolo11s-resnet50-det-v2-evaluation-01"
if RUN_NAME_TO_VIEW:
    run_dir = OUTPUT_ROOT / RUN_NAME_TO_VIEW
else:
    completed = [p.parent for p in OUTPUT_ROOT.glob("*-yolo11s-resnet50-det-v2-evaluation-*/status.json")
                 if json.loads(p.read_text()).get("status") == "complete"]
    assert completed, "완료된 전체 e2e 실행이 없습니다. 실행 로그를 확인하세요."
    run_dir = sorted(completed, key=lambda p: p.name)[-1]
print("확인하는 실행:", run_dir.name)
scores = json.loads((run_dir / "scores.json").read_text())
rows = [{"scope": key, **{k:v for k,v in value.items() if k != "per_class"}}
        for key, value in scores.items() if value is not None and key in {"all_val_detector", "all_val_final", "detector_only_reference"}]
table = pd.DataFrame(rows)
display(table[["scope", "threshold_on", "num_images", "num_gt_classes", "map_75_95", "ap_95"]])
# 전체 val의 탐지/최종 threshold 조건을 구분합니다.
plot_table = table[table["scope"] != "detector_only_reference"].set_index("scope")
ax = plot_table[["map_75_95", "ap_95"]].plot.bar(figsize=(12, 5), ylim=(0, 1.05), rot=20)
ax.set_ylabel("Reference COCO AP (0–1)")
ax.set_title("YOLO11s + shared ResNet50: class-aware e2e")
ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.savefig(run_dir / "e2e_metrics.png", dpi=180, bbox_inches="tight")
plt.show()
report_path = run_dir / "notion_e2e_report.md"
report_text = report_path.read_text()
if "![e2e score](e2e_metrics.png)" not in report_text:
    report_text += "\n![e2e score](e2e_metrics.png)\n"
    report_path.write_text(report_text)
display(Markdown(report_text))
files.download(str(report_path))
files.download(str(run_dir / "e2e_metrics.png"))


## 6. 예측 약품명과 낮은 AP 클래스 확인
예측 행에는 약품 category_id와 탐지 score·분류 확률·최종 score가 기록돼 있습니다. 아래 셀은 모델을 다시 불러오지 않습니다.


In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
map_path = Path("/content/drive/MyDrive/gitkk_archive/models/classifier/resnet50_baseline_v2/class_map.json")
class_map = json.loads(map_path.read_text())
names = {row["category_id"]: row["name"] for row in class_map["classes"]}
predictions = pd.read_csv(run_dir / "val_predictions.csv")
predictions["drug_name"] = predictions["category_id"].map(names)
display(predictions.head(20))
per_class = pd.read_csv(run_dir / "per_class_all_val_detector.csv")
per_class["drug_name"] = per_class["category_id"].map(names)
display(per_class.sort_values("ap_95").head(20))
print("AP가 낮은 약품은 위치 오차와 약품 오분류를 나눠 확인해야 합니다.")


## 결과를 읽는 방법

- `all_val_detector`: 전체 2,026장, 탐지 score 0.25 이상을 분류·평가한 e2e 점수.
- `all_val_final`: 전체 2,026장, 최종 score(탐지*분류) 0.25 이상을 평가한 e2e 점수.
- `detector_only_reference`: 약품명을 무시한 위치 탐지 점수. e2e 점수와 역할이 다릅니다.

최종 score를 기준으로 하면 낮은 분류 확률의 예측이 추가로 빠질 수 있어 점수가 달라집니다. AP를 계산할 때 정답 약품과 bbox가 모두 맞아야 합니다. 예측 개수만 맞아서는 높은 e2e 점수를 얻을 수 없습니다.

**팀 비교 전 확인:** 팀 scorer의 구현과 maxDet, score threshold 위치를 맞추세요. 현재 reference 점수가 공식 대회 채점과 완전히 동일하다고 단정하지 않습니다. 기존 SSDLite/Faster R-CNN도 공통 분류기, 같은 crop 전처리, 같은 최종 score 정의로 평가해야 합니다.

**재개:** 설치/저장소 준비/Drive 연결 후 설정 셀에 기존 E2E_RUN_NAME을 넣고 전체 실행 셀을 실행합니다. 완료된 이미지 cache는 재사용합니다. 결과를 보기만 한다면 5번 셀부터 실행하면 됩니다.
